In [24]:
# Install and Load necessary components
!python -m spacy download en_core_web_sm
!apt-get update
!apt-get install -y tesseract-ocr

import re
import spacy
import pytesseract
import pandas as pd
import json
import os
from PIL import Image
from spacy import displacy

# Load spaCy model
nlp = spacy.load('en_core_web_sm')
print("Setup Complete!") 

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 76.0 MB/s eta 0:00:0000:010:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease     
Hit:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:4 http://security.ubuntu.com/ubuntu jammy-security InRelease               
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease                         
Hit:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease                     
Hit:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease                 
Hit:8 http://arch

In [25]:
def extract_dates(text):
    # Support for MM/DD/YYYY, DD-MM-YYYY, Month DD, YYYY, and ISO [cite: 27, 28, 30]
    patterns = [
        r'\d{1,2}/\d{1,2}/\d{4}',
        r'\d{1,2}-\d{1,2}-\d{4}',
        r'\d{4}-\d{2}-\d{2}',
        r'\b(?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)[a-z]* \d{1,2},? \d{4}'
    ]
    dates = []
    for pattern in patterns:
        matches = re.findall(pattern, text)
        dates.extend(matches)
    return dates 

def extract_amounts(text):
    # Pattern for currency with optional symbols and decimals [cite: 40]
    pattern = r'\$?\d{1,3}(?:,\d{3})*(?:\.\d{2})?'
    amounts = re.findall(pattern, text)
    cleaned = []
    for amount in amounts:
        # Clean symbols to convert to float [cite: 42, 46]
        clean = amount.replace('$', '').replace(',', '')
        try:
            cleaned.append(float(clean))
        except:
            continue
    return cleaned 

def extract_invoice_number(text):
    # Patterns for common ID formats [cite: 57, 58]
    patterns = [
        r'INV-\d{4}-\d{3}',
        r'ORDER-[A-Z0-9]+',
        r'#\d{5,}'
    ]
    for pattern in patterns:
        match = re.search(pattern, text, re.IGNORECASE)
        if match:
            return match.group(0) 
    return None

In [26]:
def extract_entities(text):
    doc = nlp(text)
    entities = {
        'persons': [],
        'organizations': [],
        'locations': [],
        'dates': [],
        'money': []
    }
    
    for ent in doc.ents:
        if ent.label_ == 'PERSON':
            entities['persons'].append(ent.text) 
        elif ent.label_ == 'ORG':
            entities['organizations'].append(ent.text) 
        elif ent.label_ in ['GPE', 'LOC']:
            entities['locations'].append(ent.text) 
        elif ent.label_ == 'DATE':
            entities['dates'].append(ent.text) 
        elif ent.label_ == 'MONEY':
            entities['money'].append(ent.text) 
    return entities

In [27]:
def process_invoice(image_path):
    # Step 1: OCR [cite: 103]
    img = Image.open(image_path)
    text = pytesseract.image_to_string(img)
    
    # Step 2: Regex Extraction [cite: 107]
    invoice_data = {
        'invoice_number': extract_invoice_number(text),
        'dates': extract_dates(text),
        'amounts': extract_amounts(text)
    }
    
    # Step 3: NER Extraction [cite: 106, 113]
    entities = extract_entities(text)
    invoice_data.update(entities)
    
    # Step 4: Post-processing [cite: 116]
    if invoice_data['amounts']:
        invoice_data['total_amount'] = max(invoice_data['amounts']) 
    if invoice_data['dates']:
        invoice_data['invoice_date'] = invoice_data['dates'][0] 
        
    return invoice_data, text

# Run on Kaggle Dataset
dataset_root = "/kaggle/input/datasets/trainingdatapro/ocr-receipts-text-detection/"
df = pd.read_csv(os.path.join(dataset_root, 'receipts.csv'))

final_results = []

# Process first 3 samples as required [cite: 137]
for i in range(3):
    img_rel_path = df.iloc[i]['image_name']
    full_path = os.path.join(dataset_root, img_rel_path)
    
    if os.path.exists(full_path):
        data, raw_text = process_invoice(full_path)
        final_results.append(data)
        
        # Visualize one sample [cite: 92]
        if i == 0:
            print(f"--- Visualization for {img_rel_path} ---")
            displacy.render(nlp(raw_text), style='ent', jupyter=True)

# Save to JSON [cite: 124, 125]
with open('extracted_data.json', 'w') as f:
    json.dump(final_results, f, indent=2)



--- Visualization for images/0.jpg ---
